### Kaggle Competition: Predicting Irrigation Requirement

Author: Imaad H.

- The dataset is available at https://www.kaggle.com/competitions/predicting-irrigation-requirement/data



## About the Attributes -


### Numerical features (11)

| Column | Range | Description |
| --- | --- | --- |
| `Soil_pH` | 5–8 | Soil acidity level |
| `Soil_Moisture` | 0–70% | Water content in soil |
| `Organic_Carbon` | 0.5–1.5 | Soil fertility indicator |
| `Electrical_Conductivity` | 0–3.5 | Salt/nutrient level |
| `Temperature_C` | 10–45°C | Temperature affecting evaporation |
| `Humidity` | 20–100% | Air moisture level |
| `Rainfall_mm` | 0–3000 mm | Natural water supply |
| `Sunlight_Hours` | 4–12 hrs | Sun exposure |
| `Wind_Speed_kmh` | 0–20 km/h | Evaporation factor |
| `Field_Area_hectare` | 0–15 ha | Field size |
| `Previous_Irrigation_mm` | 0–150 mm | Past irrigation amount |

---

### Categorical features (8)

| Column | Values | Description |
| --- | --- | --- |
| `Crop_Type` | Cotton, Maize, Potato, Rice, Sugarcane, Wheat | Crop type |
| `Region` | Central, East, North, South, West | Location |
| `Soil_Type` | Clay, Loamy, Sandy, Silt | Soil type |
| `Crop_Growth_Stage` | Flowering, Harvest, Sowing, Vegetative | Growth stage |
| `Irrigation_Type` | Canal, Drip, Rainfed, Sprinkler | Irrigation method |
| `Water_Source` | Groundwater, Rainwater, Reservoir, River | Water source |
| `Season` | Kharif, Rabi, Zaid | Farming season |
| `Mulching_Used` | Yes, No | Mulching usage |

---

### Target variable

| Column | Values | Description |
| --- | --- | --- |
| `Irrigation_Need` | Low, Medium, High | Irrigation requirement level |

---

### Summary

| Type | Count |
| --- | ---: |
| Numerical | 11 |
| Categorical | 8 |
| Target | 1 |

In [ ]:
import polars as pl
import numpy as np
import plotly.express as pxi
from skimpy import skim_get_figure
from pred_irr_comp.custom.skim_extra import skim_get_figure_sized
from typing import Mapping
import polars.selectors as cs
import plotly.graph_objects as go
from plotly.graph_objects import Figure
from plotly.subplots import make_subplots

In [ ]:
# Config
_STR_DTYPES = (pl.Utf8, pl.String)
ID_COLS = ("id",)
TARGET_COL = "Irrigation_Need"
_NUM_DTYPES = (pl.Float64, pl.Int64)

### Function for reading the Dataframe

In [ ]:
# Read Dataframe
def read_dataframe(path: str) -> pl.DataFrame:
    return pl.read_csv(path)

def get_memory_size(df: pl.DataFrame) -> float:
    return df.estimated_size("mb")

### Training Dataframe

In [ ]:
train_df=read_dataframe('../data/raw/train.csv')
train_df.head()

In [ ]:
mem = get_memory_size(train_df)
print(f"Memory size of the dataframe: {mem:.2f} MB")

### About each attribute or Summary of the Dataframe

In [ ]:
# Raise console_width if long names still wrap; not matplotlib figsize — Rich uses character columns.
skim_get_figure_sized(
    train_df.drop(*ID_COLS),
    "../reports/figures/summary_table.svg",
    console_width=145,
    format="svg",
)

In [ ]:
# Gather categorical (string) columns, excluding id cols
categorical_cols = [name for name, dtype in train_df.schema.items() 
                    if dtype in _STR_DTYPES and name.lower() not in ID_COLS]

# Gather numeric columns, excluding id cols
numerical_cols = [name for name, dtype in train_df.schema.items()
                  if dtype in _NUM_DTYPES and name.lower() not in ID_COLS]

# Infer target column as last string column (by order) if possible, else raise error
if not categorical_cols:
    raise ValueError("No categorical (string) columns found in the dataframe.")

# Remove target column from categorical columns, if present
categorical_cols = [col for col in categorical_cols if col != TARGET_COL]

print(f"There are {len(numerical_cols)} numerical columns: {numerical_cols}")
print(f"There are {len(categorical_cols)} categorical columns: {categorical_cols}")
print(f"Target column: {TARGET_COL}")
print(f"ID columns: {ID_COLS}")

## Exploratory Data Analysis - Numerical Columns


In [ ]:
def _gaussian_kde_1d(
    x: np.ndarray,
    n_points: int,
    *,
    max_samples: int | None = None,
    rng_seed: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """1D Gaussian KDE with Scott's rule bandwidth h = 1.059 * sigma * n^{-1/5}.

    If ``h`` is zero or non-finite, uses ``span(x) / 10`` (or 1.0 if span is zero).

    When ``max_samples`` is set and ``len(x)`` exceeds it, subsamples without replacement.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 2:
        raise ValueError('Need at least 2 finite values for KDE')
    if max_samples is not None and x.size > max_samples:
        rng = np.random.default_rng(rng_seed)
        x = rng.choice(x, size=max_samples, replace=False)
    n = x.size
    sigma = float(np.std(x, ddof=1)) if n > 1 else 0.0
    h = 1.059 * sigma * (n ** (-1.0 / 5.0))
    if h <= 0 or not np.isfinite(h):
        span = float(np.ptp(x))
        h = span / 10.0 if span > 0 else 1.0
    x_min, x_max = float(np.min(x)), float(np.max(x))
    pad = (x_max - x_min) * 0.05 or 1.0
    xs = np.linspace(x_min - pad, x_max + pad, n_points)
    u = (xs[:, None] - x[None, :]) / h
    kernel = np.exp(-0.5 * u**2) / np.sqrt(2 * np.pi)
    density = kernel.mean(axis=1) / h
    return xs, density


def numeric_kde_figure(
    df: pl.DataFrame,
    *,
    drop_cols: tuple[str, ...] = ('id',),
    ncols: int = 3,
    default_bins: int = 80,
    nbins: Mapping[str, int] | None = None,
    kde_points: int = 256,
    kde_max_samples: int = 8000,
    vertical_spacing: float = 0.08,
    horizontal_spacing: float = 0.06,
    show_histogram: bool = True,
    show_kde: bool = True,
) -> Figure:
    """Histogram (probability density) and/or Gaussian KDE for each numeric column."""
    if not show_histogram and not show_kde:
        raise ValueError('At least one of show_histogram or show_kde must be True')

    num = df.drop(*drop_cols).select(cs.numeric())
    cols = list(num.columns)
    if not cols:
        raise ValueError('No numeric columns after dropping %s' % (drop_cols,))
    # nbins is a dictionary of column names and the number of bins to use for each column - Customizable otherwise taking default
    
    nbins_map = dict(nbins) if nbins else {}
    nrows = (len(cols) + ncols - 1) // ncols
    palette = (
        '#636efa',
        '#ef553b',
        '#00cc96',
        '#ab63fa',
        '#ffa15a',
        '#19d3f3',
        '#ff6692',
        '#b6e880',
        '#ff97ff',
        '#fecb52',
    )

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=cols,
        vertical_spacing=vertical_spacing,
        horizontal_spacing=horizontal_spacing,
    )

    for i, c in enumerate(cols):
        r = i // ncols + 1
        col_idx = i % ncols + 1
        color = palette[i % len(palette)]
        arr = num.get_column(c).to_numpy()
        arr = arr[np.isfinite(arr)]
        bins = nbins_map.get(c, default_bins)

        if show_histogram:
            fig.add_trace(
                go.Histogram(
                    x=arr,
                    nbinsx=bins,
                    histnorm='probability density',
                    name=c,
                    showlegend=False,
                    marker=dict(color=color, line=dict(width=0)),
                ),
                row=r,
                col=col_idx,
            )
        if show_kde and arr.size >= 2:
            kx, ky = _gaussian_kde_1d(
                arr,
                kde_points,
                max_samples=kde_max_samples,
            )
            fig.add_trace(
                go.Scatter(
                    x=kx,
                    y=ky,
                    mode='lines',
                    line=dict(color=color, width=2),
                    showlegend=False,
                    name=c,
                ),
                row=r,
                col=col_idx,
            )

    parts: list[str] = []
    if show_histogram:
        parts.append('histogram')
    if show_kde:
        parts.append('Kernel Density Estimation')
    title = 'Numeric column distributions (' + ' + '.join(parts) + ')'

    fig.update_layout(
        height=max(320, 260 * nrows),
        title_text=title,
    )
    fig.update_xaxes(matches=None)
    return fig

In [ ]:
fig = numeric_kde_figure(train_df, drop_cols=ID_COLS, show_kde=False,default_bins=60, nbins={})
fig.show()

In [ ]:
fig = numeric_kde_figure(train_df, drop_cols=ID_COLS, show_histogram=False)
fig.show()

### Boxplot

In [ ]:
def numeric_boxplot_figure(
    df,
    drop_cols=None,
    ncols=3,
    max_points=50000,
    vertical_spacing=0.08,
    horizontal_spacing=0.06,
    title='Numeric boxplots'
):
    """
    Create a subplot figure of boxplots for all numeric columns in the dataframe.
    
    Args:
        df: The input polars DataFrame.
        drop_cols: Columns to drop before plotting (list or None).
        ncols: Number of columns in subplot grid.
        max_points: Maximum number of points per boxplot (for speed).
        vertical_spacing: Vertical spacing between subplots.
        horizontal_spacing: Horizontal spacing between subplots.
        title: Figure title.
        
    Returns:
        a boxplot figure.
    """
    num = df
    if drop_cols is not None:
        num = num.drop(*drop_cols)
    num = num.select(cs.numeric())
    cols = list(num.columns)
    nrows = (len(cols) + ncols - 1) // ncols

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=cols,
        vertical_spacing=vertical_spacing,
        horizontal_spacing=horizontal_spacing,
    )

    for i, c in enumerate(cols):
        y = num[c].drop_nulls()
        if len(y) > max_points:
            y = y.sample(max_points, seed=0)
        r, col_i = i // ncols + 1, i % ncols + 1
        fig.add_trace(go.Box(y=y, name=c, showlegend=False), row=r, col=col_i)

    fig.update_layout(height=280 * nrows, title_text=title)
    return fig

In [ ]:
fig = numeric_boxplot_figure(train_df, drop_cols=ID_COLS, ncols=3)
fig.show()

## Correlation Matrix

In [ ]:
# Numerical columns + label-encoded target (ordinal: Low=0, Medium=1, High=2)

LABEL_MAP = {'Low': 0, 'Medium': 1, 'High': 2}

corr_df = train_df.select(
    [pl.col(c) for c in numerical_cols] + [pl.col(TARGET_COL).replace_strict(LABEL_MAP, return_dtype=pl.Int8).alias(TARGET_COL)]
)

corr_matrix = corr_df.corr()
labels = list(corr_matrix.columns)

fig = pxi.imshow(
    corr_matrix.to_numpy(),
    x=labels,
    y=labels,
    text_auto='.3f',
    aspect='auto', 
    origin='upper',
    color_continuous_scale='RdBu_r',
    zmin=-1.0,
    zmax=1.0,
    title=f'Correlation matrix (numerical features + {TARGET_COL})',
    width=1100,
    height=1000,
)
fig.update_layout(title_x=0.5, coloraxis_colorbar=dict(title='Correlation'))
fig.update_xaxes(side='bottom', tickangle=-45)
fig.show()

In [ ]:
# Top 5 strongest (non-self) correlations from a polars DataFrame
def top_5_interactions(corr_matrix):
    arr = corr_matrix.to_numpy()
    cols = list(corr_matrix.columns)
    n = len(cols)
    pairs = [ (cols[i], cols[j], arr[i, j])
              for i in range(n) for j in range(i+1, n) ]
    return sorted(pairs, key=lambda x: abs(x[2]), reverse=True)[:5]

top_5_interactions(corr_matrix)

In [ ]:
test_df = read_dataframe('../data/raw/test.csv')

### Compare KDEs of train and test sets

In [ ]:
def numeric_kde_compare_figure(
    frames: Mapping[str, pl.DataFrame],
    *,
    drop_cols: tuple[str, ...] = ('id',),
    ncols: int = 3,
    kde_points: int = 256,
    kde_max_samples: int = 1000000,
    vertical_spacing: float = 0.08,
    horizontal_spacing: float = 0.06,
    fill: bool = True,
) -> Figure:
    """Overlay Gaussian KDEs for each numeric column across multiple frames.

    `frames` is a dict like {'train': train_df, 'test': test_df}. Only numeric
    columns present in *all* frames (after dropping `drop_cols`) are plotted.
    """
    if not frames:
        raise ValueError('frames must not be empty')

    numeric_views = {
        name: df.drop(*drop_cols).select(cs.numeric()) for name, df in frames.items()
    }
    common_cols = sorted(
        set.intersection(*(set(v.columns) for v in numeric_views.values())),
        key=list(next(iter(numeric_views.values())).columns).index,
    )
    if not common_cols:
        raise ValueError('No common numeric columns across frames')

    palette = ('#636efa', '#ef553b', '#00cc96', '#ab63fa', '#ffa15a')
    color_for = {name: palette[i % len(palette)] for i, name in enumerate(frames)}

    nrows = (len(common_cols) + ncols - 1) // ncols
    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=common_cols,
        vertical_spacing=vertical_spacing,
        horizontal_spacing=horizontal_spacing,
    )

    for i, col in enumerate(common_cols):
        r = i // ncols + 1
        c = i % ncols + 1
        for j, (name, view) in enumerate(numeric_views.items()):
            arr = view.get_column(col).to_numpy()
            arr = arr[np.isfinite(arr)]
            if arr.size < 2:
                continue
            kx, ky = _gaussian_kde_1d(arr, kde_points, max_samples=kde_max_samples)
            color = color_for[name]
            fig.add_trace(
                go.Scatter(
                    x=kx,
                    y=ky,
                    mode='lines',
                    line=dict(color=color, width=2),
                    fill='tozeroy' if fill else None,
                    opacity=0.45 if fill else 1.0,
                    name=name,
                    legendgroup=name,
                    showlegend=(i == 0),
                ),
                row=r,
                col=c,
            )

    fig.update_layout(
        height=max(320, 260 * nrows),
        title_text='Train vs test KDE (numeric features)',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0),
    )
    fig.update_xaxes(matches=None)
    return fig

In [ ]:
fig = numeric_kde_compare_figure(
    {'train': train_df, 'test': test_df},
    drop_cols=ID_COLS,
)
fig.show()

### Compare Boxplots of train and test sets

In [ ]:
def numeric_boxplot_compare_figure(
    frames: Mapping[str, pl.DataFrame],
    *,
    drop_cols: tuple[str, ...] = ('id',),
    ncols: int = 3,
    max_points: int = 50_000,
    vertical_spacing: float = 0.08,
    horizontal_spacing: float = 0.06,
    boxmean: bool | str = True,
    title: str = 'Train vs test boxplots (numeric features)',
) -> Figure:
    """
    Side-by-side boxplots per numeric column across multiple frames.

    `frames` is a dict like {'train': train_df, 'test': test_df}. Only numeric
    columns present in *all* frames (after dropping `drop_cols`) are plotted.
    Each subplot shows one box per dataset on a shared y-axis.
    """
    if not frames:
        raise ValueError('frames must not be empty')

    numeric_views = {
        name: df.drop(*drop_cols).select(cs.numeric()) for name, df in frames.items()
    }
    first_cols = list(next(iter(numeric_views.values())).columns)
    common_cols = sorted(
        set.intersection(*(set(v.columns) for v in numeric_views.values())),
        key=first_cols.index,
    )
    if not common_cols:
        raise ValueError('No common numeric columns across frames')

    palette = ('#636efa', '#ef553b', '#00cc96', '#ab63fa', '#ffa15a')
    color_for = {name: palette[i % len(palette)] for i, name in enumerate(frames)}

    nrows = (len(common_cols) + ncols - 1) // ncols
    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=common_cols,
        vertical_spacing=vertical_spacing,
        horizontal_spacing=horizontal_spacing,
    )

    for i, col in enumerate(common_cols):
        r = i // ncols + 1
        c = i % ncols + 1
        for name, view in numeric_views.items():
            y = view.get_column(col).drop_nulls()
            if len(y) > max_points:
                y = y.sample(max_points, seed=0)
            fig.add_trace(
                go.Box(
                    y=y.to_numpy(),
                    name=name,
                    legendgroup=name,
                    showlegend=(i == 0),
                    marker=dict(color=color_for[name]),
                    line=dict(color=color_for[name]),
                    boxmean=boxmean,
                    boxpoints='outliers',
                    width=0.6
                ),
                row=r,
                col=c,
            )

    fig.update_layout(
        height=max(320, 280 * nrows),
        title_text=title,
        boxmode='group',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0),
    )
    return fig

In [ ]:
fig = numeric_boxplot_compare_figure(
    {'train': train_df, 'test': test_df},
    drop_cols=ID_COLS,
    ncols=2,
)
fig.update_layout(height=500 * ((len(numerical_cols) + 1) // 2), width=1400)
fig.show()

## EDA - CATEGORICAL COLUMNS


In [ ]:
# Distribution of the string columns as well

cat_df = train_df.drop(*ID_COLS).select(cs.string())
cols = list(cat_df.columns)

# Tables
for c in cols:
    print(c)
    print(cat_df[c].value_counts().sort('count', descending=True))
    print()

# Plot
ncols = 3
nrows = (len(cols) + ncols - 1) // ncols
fig = make_subplots(
    rows=nrows,
    cols=ncols,
    subplot_titles=cols,
    vertical_spacing=0.14,
    horizontal_spacing=0.06,
)

for i, c in enumerate(cols):
    vc = cat_df[c].value_counts().sort('count', descending=True)
    r = i // ncols + 1
    j = i % ncols + 1
    fig.add_trace(
        go.Bar(
            x=vc[c].cast(pl.Utf8).to_list(),
            y=vc['count'].to_list(),
            name=c,
            showlegend=False,
        ),
        row=r,
        col=j,
    )

fig.update_layout(
    height=max(400, 300 * nrows),
    title_text='Categorical column distributions',
)

fig.update_xaxes(tickangle=-40)
fig.show()

### Compare Categoricla Distributuion s of train and test sets

In [ ]:
def categorical_columns_compare_figure(
    frames: Mapping[str, pl.DataFrame],
    *,
    drop_cols: tuple[str, ...] = ('id',),
    exclude_cols: tuple[str, ...] = (),
    ncols: int = 3,
    top_k: int | None = None,
    normalize: bool = True,
    vertical_spacing: float = 0.10,
    horizontal_spacing: float = 0.06,
    row_height: int = 340,
    title: str = 'Categorical distributions (train vs test)',
) -> Figure:
    """Grouped bar chart per categorical column across frames.

    - `frames`: mapping like {'train': train_df, 'test': test_df}.
    - `normalize=True` plots proportions (each dataset sums to 1 per column), so
      class balance is directly comparable.
    - `top_k` keeps only the most frequent categories (by combined count).
    """
    if not frames:
        raise ValueError('frames must not be empty')

    cat_views = {
        name: df.drop(*drop_cols).select(cs.string()) for name, df in frames.items()
    }
    first_cols = list(next(iter(cat_views.values())).columns)
    common_cols = [
        c for c in first_cols
        if c not in exclude_cols
        and all(c in v.columns for v in cat_views.values())
    ]
    if not common_cols:
        raise ValueError('No common categorical columns across frames')

    palette = ('#636efa', '#ef553b', '#00cc96', '#ab63fa', '#ffa15a')
    color_for = {name: palette[i % len(palette)] for i, name in enumerate(frames)}

    nrows = (len(common_cols) + ncols - 1) // ncols
    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=common_cols,
        vertical_spacing=vertical_spacing,
        horizontal_spacing=horizontal_spacing,
    )

    for i, col in enumerate(common_cols):
        r = i // ncols + 1
        c = i % ncols + 1

        counts_per_frame = {
            name: (
                view.get_column(col)
                .drop_nulls()
                .value_counts()
                .sort('count', descending=True)
            )
            for name, view in cat_views.items()
        }

        combined = {}
        for name, vc in counts_per_frame.items():
            for cat, n in zip(vc[col].to_list(), vc['count'].to_list()):
                combined[cat] = combined.get(cat, 0) + int(n)
        categories = sorted(combined, key=combined.get, reverse=True)
        if top_k is not None:
            categories = categories[:top_k]

        for name, vc in counts_per_frame.items():
            lookup = dict(zip(vc[col].to_list(), vc['count'].to_list()))
            total = sum(lookup.values()) or 1
            y = [
                (lookup.get(cat, 0) / total) if normalize else lookup.get(cat, 0)
                for cat in categories
            ]
            fig.add_trace(
                go.Bar(
                    x=categories,
                    y=y,
                    name=name,
                    legendgroup=name,
                    showlegend=(i == 0),
                    marker=dict(color=color_for[name]),
                    hovertemplate='%{x}: %{y:.3f}<extra>' + name + '</extra>',
                ),
                row=r,
                col=c,
            )

        fig.update_xaxes(tickangle=-30, row=r, col=c)
        fig.update_yaxes(
            title_text='Proportion' if normalize else 'Count',
            row=r,
            col=c,
        )

    fig.update_layout(
        height=max(row_height, row_height * nrows),
        title_text=title,
        barmode='group',
        bargap=0.15,
        bargroupgap=0.05,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0),
    )
    return fig

In [ ]:
fig = categorical_columns_compare_figure(
    {'train': train_df, 'test': test_df},
    drop_cols=ID_COLS,
    exclude_cols=(TARGET_COL,),
    ncols=2,
)
fig.show()

In [ ]:
# Target distribution (Irrigation Needed) in a pie chart
fig = pxi.pie(
    train_df,
    names='Irrigation_Need',
    title='Target Distribution',
    hole=0.01
)
fig.show()